# Benchmark: Strands decider (2B) on our routing tests

**What.** A zero-shot comparison. [`StrandsAgents/strands-decider-2B-hobson-v19`](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) is a ready-made **decision model**: Qwen3.5-2B-Base with a LoRA adapter and a small readout head. Given a *state* (text) and a *choice* question with named options, it returns a calibrated probability per option. Routing is exactly such a choice: the state is the conversation, and the options are the agents with their descriptions.

**The tests**, the same files as `modernbert.ipynb`, none of which this model was trained on:

| ID | Test | Size |
|---|---|---|
| E11 | First message only: pick 1 of 5 agents | 100 messages, 10 companies |
| E10 | Realistic chat history: route the latest user turn | 35 conversations, 193 turns, 13 companies, 3 agents each |
| E3 | Banking chat history (benchmark + extended) | 225 conversations, 5 agents |

**Compared with** our best router so far (run 9: Nemotron-1B encoder + trained head) from the experiment ledger.

Nothing here is trained: the decider is used as published.

## 1 — Setup

Install the decider package, pin transformers to the same version as `modernbert.ipynb`, and show the GPU.

In [75]:
!pip uninstall -y -q torchao
!pip install -q "strands-decider" "transformers==5.17.0"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
      "| bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 18.3 MB/s eta 0:00:0000:01
torch 2.11.0+cu128 | cuda: True | Tesla T4 | bf16 supported: True


## 2 — Test data

The test files live in the repo on the experiment branch. They are cloned (or refreshed) from that branch.

In [76]:
import json, subprocess, time
from collections import Counter
from pathlib import Path

DATA_BRANCH = "exp/phase8-general-choice"
repo = Path("llms-the-hard-way")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                    "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
DATA_DIR = repo / "data" / "modernbert_router"

ft = json.load(open(DATA_DIR / "first_turn" / "messages.json"))
real_convs = json.load(open(DATA_DIR / "realistic" / "conversations.json"))
banking_all = json.load(open(DATA_DIR / "banking" / "benchmark.json")) + json.load(open(DATA_DIR / "banking" / "extended.json"))
BANKING_AGENTS = {                                          # exactly as in modernbert.ipynb, Cell 37
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
print(f"E11 first turn: {len(ft['messages'])} messages | E10 realistic: {len(real_convs)} conversations, "
      f"{sum(len(c['user_turns']) for c in real_convs)} turns | banking: {len(banking_all)} conversations")

E11 first turn: 100 messages | E10 realistic: 35 conversations, 193 turns | banking: 225 conversations


From https://github.com/skamalj/llms-the-hard-way
 * branch            exp/benchmark-strands-decider -> FETCH_HEAD
Switched to a new branch 'exp/benchmark-strands-decider'


## 3 — Load the decider

`load_engine` downloads the base model (Qwen3.5-2B-Base) and the checkpoint's adapter and head. The checkpoint asks for **bfloat16**. A T4 has no bfloat16 support, so in that case the model is moved to **float32** (about 8 GB for 2B parameters). A first question from the model card checks that it answers sensibly.

In [77]:
from strands_decider.infer import load_engine
from strands_decider.schema import ChoiceQuestion

CHECKPOINT = "StrandsAgents/strands-decider-2B-hobson-v19"
t0 = time.perf_counter()
engine = load_engine(CHECKPOINT, device="cuda:1")
if not torch.cuda.is_bf16_supported():                       # e.g. a T4: run in float32 instead of bfloat16
    engine.model.to(torch.float32)
print(f"loaded {CHECKPOINT} in {time.perf_counter() - t0:.0f}s | dtype {next(engine.model.torso.parameters()).dtype}")

demo = engine.ask("Help! My payouts have been failing for 3 days!",
                  {"team": ChoiceQuestion(instructions="Which team should handle this?",
                                          criteria={"billing": "billing", "sales": "sales", "retail": "retail"})})
print("model card example ->", demo.answers["team"].choice, demo.answers["team"].probabilities)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


loaded StrandsAgents/strands-decider-2B-hobson-v19 in 32s | dtype torch.bfloat16
model card example -> billing {'billing': 0.7634, 'sales': 0.1213, 'retail': 0.1154}


## 4 — The routing question

- **State:** the conversation as a plain transcript, one line per message, with the latest user message repeated at the end so the question is unambiguous.
- **Question:** *"Which agent should handle the user's latest message?"*
- **Options:** each agent name with its description.

`window` keeps only the last N user turns (with the replies between them), to compare full history with a recent window.

In [78]:
INSTRUCTIONS = "Which agent should handle the user's latest message?"


def state_text(messages, window=None):
    """Transcript of the conversation up to the latest user message, optionally only the last `window` user turns."""
    if window is not None:
        user_idx = [i for i, m in enumerate(messages) if m["role"] == "user"]
        messages = messages[user_idx[-window]:] if len(user_idx) > window else messages
    lines = [f"{m['role']}: {m['content']}" for m in messages]
    return "Conversation:\n" + "\n".join(lines) + f"\n\nLatest user message: {messages[-1]['content']}"


def decide(messages, agents, window=None):
    """-> (chosen agent, probabilities {agent: p}) for the latest user message."""
    q = ChoiceQuestion(instructions=INSTRUCTIONS, criteria=dict(agents))
    ans = engine.ask(state_text(messages, window), {"route": q}).answers["route"]
    probs = dict(sorted(ans.probabilities.items(), key=lambda kv: -kv[1]))
    return ans.choice, probs


print(state_text(real_convs[0]["messages"][:5]))
print(decide(real_convs[0]["messages"][:5], real_convs[0]["agents"]))

Conversation:
user: Hi, I need a refill on my blood pressure tablets.
assistant: Of course. Is it the amlodipine from Dr Shah?
user: yes that one
assistant: Done, it'll be ready tomorrow after 2pm.
user: Can I take it with ibuprofen? I've got a bad back.

Latest user message: Can I take it with ibuprofen? I've got a bad back.
('Pharmacist Advice Agent', {'Pharmacist Advice Agent': 0.8201, 'Repeat Prescriptions Agent': 0.1728, 'Home Delivery Agent': 0.0071})


## 5 — E11: first message only (100 messages, 5 agents each)

Top-1 and top-2 per message type, and every mistake. Our router (run 9) scored **93 / 100 top-1, 97 / 100 top-2**.

In [8]:
t0 = time.perf_counter()
e11 = []
for m in ft["messages"]:
    agents = ft["companies"][m["company"]]
    pick, probs = decide([{"role": "user", "content": m["message"]}], agents)
    e11.append({**m, "pick": pick, "top2": list(probs)[:2], "p": probs[pick]})
print(f"E11: {len(e11)} decisions in {time.perf_counter() - t0:.0f}s\n")

print(f"{'type':10s} | {'msgs':>4} | {'top-1':>9} | {'top-2':>9}")
for tt in ["clear", "short", "indirect", "close", "noisy", "ALL"]:
    rows = [r for r in e11 if tt == "ALL" or r["type"] == tt]
    print(f"{tt:10s} | {len(rows):>4} | {sum(r['pick'] == r['target_agent'] for r in rows):>4} / {len(rows):<3} | "
          f"{sum(r['target_agent'] in r['top2'] for r in rows):>4} / {len(rows):<3}")
print("\nmistakes:")
for r in [r for r in e11 if r["pick"] != r["target_agent"]]:
    print(f"  {r['type']:8s} | {r['message'][:60]:60s} | expected {r['target_agent'].replace(' Agent', ''):26s} | got {r['pick'].replace(' Agent', '')} ({r['p']:.2f})")

E11: 100 decisions in 28s

type       | msgs |     top-1 |     top-2
clear      |   31 |   31 / 31  |   31 / 31 
short      |   17 |   17 / 17  |   17 / 17 
indirect   |   17 |   16 / 17  |   17 / 17 
close      |   20 |   17 / 20  |   20 / 20 
noisy      |   15 |   15 / 15  |   15 / 15 
ALL        |  100 |   96 / 100 |  100 / 100

mistakes:
  close    | my garden waste bin was skipped this week                    | expected Garden Waste               | got Bin Collections (0.89)
  indirect | The courier didn't turn up for the pickup I booked.          | expected Shipping Labels            | got Disputes (0.69)
  close    | I booked the wrong day by mistake, can I swap to Friday?     | expected Refunds & Exchanges        | got Ticket Booking (0.77)
  close    | How do I give my accountant access without seeing payroll?   | expected Users & Login              | got Payroll (0.69)


## 6 — E10: realistic chat history (193 turns)

Every user turn is routed with the conversation so far, assistant replies included, in two variants:
- **full** history;
- a **window** of the last 3 user turns.

The decider has no "current agent" input, so each turn is routed independently. Our router (run 9) scored **154 / 193**, and its cross-only head **158 / 193**.

In [79]:
TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]
t0 = time.perf_counter()
e10 = []
for conv in real_convs:
    for u in conv["user_turns"]:
        prefix = conv["messages"][:u["message_index"] + 1]
        full, _ = decide(prefix, conv["agents"])
        win, _ = decide(prefix, conv["agents"], window=3)
        e10.append({"id": conv["id"], "type": u["turn_type"], "expected": u["expected_agent"],
                    "message": prefix[-1]["content"], "full": full, "window 3": win})
print(f"E10: {len(e10)} turns x 2 variants in {time.perf_counter() - t0:.0f}s\n")

print(f"{'turn type':14s} | {'turns':>5} | {'full history':>12} | {'window 3':>10}")
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in e10 if tt == "ALL" or r["type"] == tt]
    if rows:
        print(f"{tt:14s} | {len(rows):>5} | {sum(r['full'] == r['expected'] for r in rows):>6} / {len(rows):<4} | "
              f"{sum(r['window 3'] == r['expected'] for r in rows):>5} / {len(rows):<4}")
print("\nfull-history mistakes (first 15):")
for r in [r for r in e10 if r["full"] != r["expected"]][:15]:
    print(f"  {r['id']:9s} {r['type']:12s} | {r['message'][:50]:50s} | expected {r['expected'].replace(' Agent', ''):22s} | got {r['full'].replace(' Agent', '')}")

E10: 193 turns x 2 variants in 119s

turn type      | turns | full history |   window 3
start          |    35 |     33 / 35   |    33 / 35  
answer         |    51 |     41 / 51   |    47 / 51  
follow         |    21 |     18 / 21   |    21 / 21  
aspect         |     7 |      7 / 7    |     7 / 7   
closing        |    15 |     11 / 15   |    12 / 15  
also_stay      |     4 |      4 / 4    |     4 / 4   
switch         |    42 |     26 / 42   |    30 / 42  
short_switch   |     6 |      2 / 6    |     3 / 6   
return         |     7 |      6 / 7    |     3 / 7   
handoff_ok     |     2 |      1 / 2    |     1 / 2   
branch         |     3 |      2 / 3    |     2 / 3   
ALL            |   193 |    151 / 193  |   163 / 193 

full-history mistakes (first 15):
  pharm-2   switch       | And is my inhaler prescription in that bag too?    | expected Repeat Prescriptions   | got Home Delivery
  pharm-2   follow       | how long does that usually take                    | expected Repeat P

## 7 — Banking chat history (225 conversations, 5 agents)

The banking test from `modernbert.ipynb` (benchmark + extended), with the last user message routed. Our router (run 9) scored **193 / 225** (benchmark **43 / 50**).

In [9]:
t0 = time.perf_counter()
bank = []
for ex in banking_all:
    pick, _ = decide(ex["messages"], BANKING_AGENTS)
    bank.append(pick == ex["target_agent"])
print(f"banking: {sum(bank[:50])} / 50 benchmark | {sum(bank)} / {len(bank)} all | {time.perf_counter() - t0:.0f}s")

banking: 48 / 50 benchmark | 214 / 225 all | 109s


## 8 — Summary next to our router

Our numbers are from the experiment ledger (run 9: Nemotron-1B encoder + trained head, `exp/phase3b-nemotron`).

In [12]:
ours = {"E11 first turn, top-1 (100)": 93, "E11 first turn, top-2 (100)": 97,
        "E10 chat history (193)": 154, "banking benchmark (50)": 43, "all banking (225)": 193}
dec = {"E11 first turn, top-1 (100)": sum(r["pick"] == r["target_agent"] for r in e11),
       "E11 first turn, top-2 (100)": sum(r["target_agent"] in r["top2"] for r in e11),
       "E10 chat history (193)": max(sum(r["full"] == r["expected"] for r in e10), sum(r["window 3"] == r["expected"] for r in e10)),
       "banking benchmark (50)": sum(bank[:50]), "all banking (225)": sum(bank)}
print(f"{'test':32s} | {'our router (run 9)':>18} | {'strands decider':>15}")
for k in ours:
    print(f"{k:32s} | {ours[k]:>18} | {dec[k]:>15}")
print("\n(E10 decider = the better of full history / window 3)")

test                             | our router (run 9) | strands decider
E11 first turn, top-1 (100)      |                 93 |              96
E11 first turn, top-2 (100)      |                 97 |             100
E10 chat history (193)           |                154 |             163
banking benchmark (50)           |                 43 |              48
all banking (225)                |                193 |             214

(E10 decider = the better of full history / window 3)


In [80]:
# Section 9 — E10 on the same split as modernbert.ipynb Cell 70: 6 test companies (83 turns), by turn type

companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])          # identical split to Cell 70
company_of = {c["id"]: c["company"] for c in real_convs}
test_rows = [r for r in e10 if company_of[r["id"]] in TEST_COMPANIES]

print(f"TEST companies: {', '.join(sorted(TEST_COMPANIES))} | {len(test_rows)} turns")
print(f"{'turn type':14s} | {'turns':>5} | {'decider full':>12} | {'decider window 3':>16}")
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in test_rows if tt == "ALL" or r["type"] == tt]
    if rows:
        print(f"{tt:14s} | {len(rows):>5} | {sum(r['full'] == r['expected'] for r in rows):>7} / {len(rows):<3} | "
              f"{sum(r['window 3'] == r['expected'] for r in rows):>10} / {len(rows):<3}")


TEST companies: cruise line, driving school, games console company, pet food subscription, photo printing service, solar installer | 83 turns
turn type      | turns | decider full | decider window 3
start          |    15 |      14 / 15  |         14 / 15 
answer         |    22 |      17 / 22  |         20 / 22 
follow         |     9 |       8 / 9   |          9 / 9  
aspect         |     1 |       1 / 1   |          1 / 1  
closing        |     5 |       4 / 5   |          5 / 5  
also_stay      |     3 |       3 / 3   |          3 / 3  
switch         |    20 |      10 / 20  |         13 / 20 
short_switch   |     4 |       1 / 4   |          2 / 4  
return         |     2 |       2 / 2   |          1 / 2  
branch         |     2 |       2 / 2   |          1 / 2  
ALL            |    83 |      62 / 83  |         69 / 83 


## 10 — E12: grounded choice questions (phase 8)

**What.** The decider on its home ground: general questions about a given text, in its **native** question types. These are the same 170 items as `modernbert.ipynb` Cell 72 (`data/modernbert_router/choice_test/items.json`). Every answer is in the text; none needs world knowledge.

| E12 type | Decider question | Read back as |
|---|---|---|
| yes / no (58) | `NoulQuestion(instructions=question)` | P(true) > 0.5 → "yes" |
| choice (59) | `ChoiceQuestion(instructions=question, criteria={option: option})`, as in the model-card example | the chosen option |
| score (53) | `ScoreQuestion(instructions=question, criteria=levels from low to high)` | the most likely level, and the expected level (its `score`) |

**Expect.** Pass / total per type and per category, score errors in levels, the question-group check (same text, different questions), every mistake, and the time per question. It runs after sections 1–3 only (setup, data, load).


In [ ]:
# Section 10 — E12 with the decider in its native question types
from strands_decider.schema import NoulQuestion, ScoreQuestion

e12_items = json.load(open(DATA_DIR / "choice_test" / "items.json"))["items"]


def ask_item(it):
    """-> (pick, probabilities in option order, expected level or None)."""
    if it["type"] == "noul":
        p_true = engine.ask(it["state"], {"q": NoulQuestion(instructions=it["question"])}).answers["q"].noul
        return ("yes" if p_true > 0.5 else "no"), [p_true, 1 - p_true], None
    if it["type"] == "choice":
        ans = engine.ask(it["state"], {"q": ChoiceQuestion(instructions=it["question"],
                                                            criteria={o: o for o in it["options"]})}).answers["q"]
        return ans.choice, [ans.probabilities.get(o, 0.0) for o in it["options"]], None
    ans = engine.ask(it["state"], {"q": ScoreQuestion(instructions=it["question"], criteria=it["options"])}).answers["q"]
    # probabilities are keyed by level name or by level index, depending on the version
    probs = [ans.probabilities.get(o, ans.probabilities.get(str(i), 0.0)) for i, o in enumerate(it["options"])]
    top = max(range(len(probs)), key=lambda i: probs[i]) if any(probs) else min(len(it["options"]) - 1, max(0, round(ans.score)))
    return it["options"][top], probs, float(ans.score)


ask_item(e12_items[0])                                    # warm-up (not timed)
e12_rows, t0 = [], time.perf_counter()
for it in e12_items:
    pick, probs, expected = ask_item(it)
    row = {**it, "pick": pick, "probs": probs, "ok": pick == it["answer"]}
    if it["type"] == "score":
        row["expected_level"] = expected
        row["level_error"] = abs(it["options"].index(pick) - it["answer_index"])
    e12_rows.append(row)
e12_ms = (time.perf_counter() - t0) * 1000 / len(e12_items)
print(f"E12: {len(e12_rows)} items, {e12_ms:.0f} ms per question\n")

print(f"{'type':8s} | {'items':>5} | {'correct':>9}")
for t in ["noul", "choice", "score", "ALL"]:
    rows = [r for r in e12_rows if t == "ALL" or r["type"] == t]
    print(f"{t:8s} | {len(rows):>5} | {sum(r['ok'] for r in rows):>4} / {len(rows):<4}")
sc = [r for r in e12_rows if r["type"] == "score"]
print(f"score: exact level {sum(r['level_error'] == 0 for r in sc)} / {len(sc)} | within 1 level {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)} | "
      f"mean |expected - true| {sum(abs(r['expected_level'] - r['answer_index']) for r in sc) / len(sc):.2f} levels")
yes_rows = [r for r in e12_rows if r["type"] == "noul"]
print(f"yes / no: answered yes {sum(r['pick'] == 'yes' for r in yes_rows)} of {len(yes_rows)} (truth: {sum(r['answer'] == 'yes' for r in yes_rows)} yes)")

print("\nper category (all types):")
for cat in sorted({r["category"] for r in e12_rows}):
    rows = [r for r in e12_rows if r["category"] == cat]
    print(f"  {cat:10s} {sum(r['ok'] for r in rows):>3} / {len(rows)}")

groups = {}
for r in e12_rows:
    if "pair" in r:
        groups.setdefault(r["pair"], []).append(r)
print(f"\nquestion groups (same text, different questions): all answers right in {sum(all(r['ok'] for r in g) for g in groups.values())} of {len(groups)} groups")
for pid, g in groups.items():
    print(f"  {pid}: " + " | ".join(f"{r['question'][:34]} -> {r['pick']}{'' if r['ok'] else ' x'}" for r in g))

print("\nmistakes:")
for r in [r for r in e12_rows if not r["ok"]]:
    s_ = r["state"] if len(r["state"]) <= 46 else r["state"][:43] + "..."
    print(f"  {r['id']} {r['type']:6s} | {s_:46s} | {r['question'][:38]:38s} | expected {r['answer']:14s} | got {r['pick']}")

with open("e12_decider.json", "w") as fh:
    json.dump({"ms_per_question": e12_ms, "rows": [{k: r[k] for k in ("id", "type", "category", "pick", "ok", "probs")} for r in e12_rows]}, fh)
